In [ ]:
def shift_features_2_candle(data):
    for i in ['open', 'close', 'low', 'high', 'volume', 'pattern']:
        data[f"{i}_N"] = data[i]
        data[f'{i}_N-1'] = data[i].shift(1)
    data.drop(['open', 'close', 'low', 'high', 'volume', 'pattern'], axis=1, inplace=True)
    data.dropna(inplace=True)
    for i in ['open', 'close', 'low', 'high', 'volume', 'pattern']:
        data[f'{i}_N-1'] = data[f'{i}_N-1'].astype('int32')
    return data


In [ ]:
def features_of_candle(data):
    # 1. Размер тела
    data['body_N'] = data['close_N'] - data['open_N']
    data['body_N-1'] = data['close_N-1'] - data['open_N']
    
    # 2. Размер всей свечи (high - low)
    data['candle_height_N'] = data['high_N'] - data['low_N']
    data['candle_height_N-1'] = data['high_N-1'] - data['low_N-1']
    
    # 3. Верхняя тень
    data['shadow_up_N'] = np.where(
        data['open_N'] < data['close_N'], 
        data['high_N'] - data['close_N'],
        data['high_N'] - data['open_N'])
    
    data['shadow_up_N-1'] = np.where(
        data['open_N-1'] < data['close_N-1'], 
        data['high_N-1'] - data['close_N-1'],
        data['high_N-1'] - data['open_N-1'])
    
    # 4. Нижняя тень
    data['shadow_down_N'] = np.where(
        data['open_N'] < data['close_N'],
        data['open_N'] - data['low_N'],
        data['close_N'] - data['low_N'])
    
    data['shadow_down_N-1'] = np.where(
        data['open_N-1'] < data['close_N-1'],
        data['open_N-1'] - data['low_N-1'],
        data['close_N-1'] - data['low_N-1'])
    
    # 5. Разница между телами (body_N - body_N-1)
    data['body_diff'] = data['body_N'] - data['body_N-1']
    
    # 6. Сумма теней
    data['shadow_sum_N'] = data['shadow_up_N'] + data['shadow_down_N']
    data['shadow_sum_N-1'] = data['shadow_up_N-1'] + data['shadow_down_N-1']
    
    # 7. Отношение верхней тени к телу (логически правильный метод с тремя случаями)
    data['ratio_up_shadow_to_body_N'] = np.where(
    data['body_N'] != 0,
    data['shadow_up_N'] / data['body_N'],
    np.where(
        data['shadow_up_N'] == 0,
        0,
        100))

    data['ratio_up_shadow_to_body_N_abs'] = np.where(
    data['body_N'] != 0,
    np.abs(data['shadow_up_N'] / data['body_N']),
    np.where(
        data['shadow_up_N'] == 0,
        0,
        100))

    data['ratio_up_shadow_to_body_N-1'] = np.where(
    data['body_N-1'] != 0,
    data['shadow_up_N-1'] / data['body_N-1'],
    np.where(
        data['shadow_up_N-1'] == 0,
        0,
        100))

    data['ratio_up_shadow_to_body_N-1_abs'] = np.where(
    data['body_N-1'] != 0,
    np.abs(data['shadow_up_N-1'] / data['body_N-1']),
    np.where(
        data['shadow_up_N-1'] == 0,
        0,
        100))


    # 8. Отношение нижней тени ко всему телу по модулу и просто
    data['ratio_down_shadow_to_body_N'] = np.where(
    data['body_N'] != 0,
    data['shadow_down_N'] / data['body_N'],
    np.where(
        data['shadow_down_N'] == 0,
        0,
        100))

    data['ratio_down_shadow_to_body_N_abs'] = np.where(
    data['body_N'] != 0,
    np.abs(data['shadow_down_N'] / data['body_N']),
    np.where(
        data['shadow_down_N'] == 0,
        0,
        100))

    data['ratio_down_shadow_to_body_N-1'] = np.where(
    data['body_N-1'] != 0,
    data['shadow_down_N-1'] / data['body_N-1'],
    np.where(
        data['shadow_down_N-1'] == 0,
        0,
        100))

    data['ratio_down_shadow_to_body_N-1_abs'] = np.where(
    data['body_N-1'] != 0,
    np.abs(data['shadow_down_N-1'] / data['body_N-1']),
    np.where(
        data['shadow_down_N-1'] == 0,
        0,
        100))
    
    # 9. Отношение суммы теней ко всему телу по модулу и просто
    data['ratio_sum_shadow_to_body_N'] = np.where(
    data['body_N'] != 0,
    data['shadow_sum_N'] / data['body_N'],
    np.where(
        data['shadow_sum_N'] == 0,
        0,
        100))

    data['ratio_sum_shadow_to_body_N_abs'] = np.where(
    data['body_N'] != 0,
    np.abs(data['shadow_sum_N'] / data['body_N']),
    np.where(
        data['shadow_sum_N'] == 0,
        0,
        100))

    data['ratio_sum_shadow_to_body_N-1'] = np.where(
    data['body_N-1'] != 0,
    data['shadow_sum_N-1'] / data['body_N-1'],
    np.where(
        data['shadow_sum_N-1'] == 0,
        0,
        100))

    data['ratio_sum_shadow_to_body_N-1_abs'] = np.where(
    data['body_N-1'] != 0,
    np.abs(data['shadow_sum_N-1'] / data['body_N-1']),
    np.where(
        data['shadow_sum_N-1'] == 0,
        0,
        100))
    
    # 10. Насколько % одна свеча перекрывает другую. Будем смотреть относительно тела N
    data['ratio_of_body'] = np.where(((data['body_N'] != 0) &  (data['body_N-1'] != 0)), 
                                     data['body_N'] / data['body_N-1'], 0)
                                     
    data['ratio_of_body_abs'] = np.where(((data['body_N'] != 0) &  (data['body_N-1'] != 0)), 
                                     np.abs(data['body_N'] / data['body_N-1']), 0)
    
    # 11. Флаг, если тело = 0 
    data['is_zero_body_N'] = (data['body_N'] == 0).astype(int)
    data['is_zero_body_N-1'] = (data['body_N-1'] == 0).astype(int)

    return data
